In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

## Helper functions

In [2]:

from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

## Tool Setup

**Schema**

- We're just overriding some values of the built-in schema

In [4]:
web_search_schema = {
    "type": "web_search_20260209",
    "name": "web_search",
    "max_uses": 5,
}

## Test Runs

**Without Websearch Tool**

In [ ]:
# messages = []
# add_user_message(
#     messages,
#     """
# What is the current stock price of ZGLD?
#     """,
# )
# response = chat(messages)
# response

Message(id='msg_015Kqw6C348eFiMmNKZCCVjf', container=None, content=[TextBlock(citations=None, text='I don\'t have access to real-time stock market data or current stock prices. To find the current stock price of ZGLD (which appears to be a gold-related ETF or security), I recommend:\n\n1. **Financial websites**: Check Yahoo Finance, Google Finance, MarketWatch, or Bloomberg\n2. **Your brokerage account**: If you have one, it will show real-time or near-real-time prices\n3. **Search engines**: Simply searching "ZGLD stock price" will typically show the current price\n\nIf you let me know what ZGLD represents (it might be a ticker symbol for a specific exchange or region), I could provide more context about the security itself, though I still wouldn\'t be able to provide the live price.', type='text')], model='claude-sonnet-4-5-20250929', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, eph

**With Websearch**

- Provides the query it uses to search in `server_tool_use` block.
- It also provides the `citations` list

In [11]:
import json

# Without Tool
messages = []
add_user_message(
    messages,
    """
What is the current stock price of ZGLD?
    """,
)
response = chat(messages, tools=[web_search_schema])

# Print only text content from response
for block in response.content:
    if block.type == "text":
        print(block.text)

print("=" * 20)

# Pretty print raw response content - For understanding the structure of response and for debugging
raw_content = [
    b.model_dump() if hasattr(b, "model_dump") else str(b)
    for b in response.content
]
print("Raw response (pretty):")
print(json.dumps(raw_content, indent=2, ensure_ascii=False))

I'll search for the current stock price of ZGLD for you.
Based on the search results, 
ZGLD trades at 68.46 CAD today
. 

ZGLD is the 
BMO Gold Bullion ETF issued by BMO Asset Management Inc and is listed and traded on the Toronto Stock Exchange
. The price shown is in Canadian dollars (CAD).
Raw response (pretty):
[
  {
    "citations": null,
    "text": "I'll search for the current stock price of ZGLD for you.",
    "type": "text"
  },
  {
    "id": "srvtoolu_01V39QZD3EiZWSBwU33nb5DY",
    "caller": {
      "type": "direct"
    },
    "input": {
      "code": "\nimport json\nresult = await web_search({\"query\": \"ZGLD stock price today\"})\nparsed = json.loads(result)\n\n# Check if there's an error\nif parsed and len(parsed) > 0 and parsed[0].get(\"type\") == \"error\":\n    print(f\"Error: {parsed[0].get('error_code')}\")\nelse:\n    # Print key information from search results\n    for i, item in enumerate(parsed[:3]):  # Look at top 3 results\n        print(f\"\\n--- Result {i} --

**With Websearch - targetted/restricted domains**

In [12]:
web_search_schema = {
    "type": "web_search_20260209",
    "name": "web_search",
    "max_uses": 5,
    "allowed_domains": ["finance.yahoo.com"],
}

In [13]:
import json

# Without Tool
messages = []
add_user_message(
    messages,
    """
What is the current stock price of ZGLD?
    """,
)
response = chat(messages, tools=[web_search_schema])

# Print only text content from response
for block in response.content:
    if block.type == "text":
        print(block.text)

print("=" * 20)

# Pretty print raw response content - For understanding the structure of response and for debugging
raw_content = [
    b.model_dump() if hasattr(b, "model_dump") else str(b)
    for b in response.content
]
print("Raw response (pretty):")
print(json.dumps(raw_content, indent=2, ensure_ascii=False))

I'll search for the current stock price of ZGLD.
Based on the search results, I can see there are actually multiple securities with the ticker ZGLD:

1. **ZGLD.TO (BMO Gold Bullion ETF - CAD Units)**: 
The most recent price shown is around $73.38-$74.03 CAD
, trading on the Toronto Stock Exchange.

2. **ZGLD.SW (Swisscanto (CH) Gold ETF)**: This is a Swiss gold ETF trading in CHF.

3. **ZGLD-U.TO (BMO Gold Bullion ETF - USD Units)**: 
Trading around $63.18-$63.26 USD
.

The most commonly referenced ZGLD is the **BMO Gold Bullion ETF (CAD Units)** trading on the Toronto Stock Exchange. Could you please clarify which specific ZGLD you're interested in, or if you meant the Canadian-listed BMO Gold Bullion ETF?
Raw response (pretty):
[
  {
    "citations": null,
    "text": "I'll search for the current stock price of ZGLD.",
    "type": "text"
  },
  {
    "id": "srvtoolu_01XN3STCjmoTUA7WeKzTWRQF",
    "caller": {
      "type": "direct"
    },
    "input": {
      "code": "\nimport json\n\